# 05 — Baseline Forecasts

## 1. Introduction

**Forecasting task.** At 12:00 (Swiss local time) on day $t$, an electricity procurement team needs a forecast of the total electricity consumption in Basel-Stadt on the next day $d = t + 1$, in MWh/day. Example: to forecast 17 November 2026, the forecast must be issued by 12:00 on 16 November 2026.

**Data availability.** Consumption data are published with a delay and are sometimes revised. Following the professor-approved conservative assumption, we treat only consumption observations from day $d-5$ and older as known at the cutoff (`lag_5` and older). This is an agreed assumption, not a verified publication timestamp. All lags are counted in calendar days relative to the target date $d$.

**Why a baseline?** A forecasting model is only useful if it beats what a planner could do without any model. A simple, transparent benchmark turns every other model's error into a meaningful comparison: we report how much each model improves on the baseline, on exactly the same dates. A complex model that cannot beat a one-line rule adds complexity without adding value, and a model that beats it by an implausibly large margin is a warning sign for data leakage.

**Why the same weekday one week earlier?** Electricity consumption has a strong weekly pattern (workdays vs. weekends, see `03_data_exploration.ipynb`) and changes only gradually with the seasons. The same weekday one week earlier captures both the weekly pattern and the current consumption level. Under the five-day rule, `lag_7` is the most recent observation that falls on the same weekday as the target day (`lag_5` and `lag_6` are permitted, but fall on different weekdays).

**Baseline rule.**

$$\hat y_d^{\,baseline} = y_{d-7}$$

If $y_{d-7}$ is unavailable, we use the next older same-weekday observation: $y_{d-14}$, then $y_{d-21}$, then $y_{d-28}$.

**Evaluation.** The baseline has no parameters to estimate, so it requires no training. We evaluate it on the 2025 validation period only, using MAE (primary) and RMSE (secondary), both in MWh/day. The 2026 holdout rows are loaded for structural checks only and are not evaluated in this notebook.

In [1]:
# Setup: imports, project paths and methodology constants
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    """Walk upwards from `start` until the folder containing data/processed/features.csv is found."""
    for folder in [start, *start.parents]:
        if (folder / "data" / "processed" / "features.csv").exists():
            return folder
    raise FileNotFoundError(
        f"data/processed/features.csv not found above {start}. "
        "Check that you opened the repository, or run code/04_prepare_features.py first."
    )


ROOT = find_repo_root(Path.cwd().resolve())
FEATURES_PATH = ROOT / "data" / "processed" / "features.csv"
FIG_DIR = ROOT / "output" / "figures"
TABLE_DIR = ROOT / "output" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Methodology constants (must match 04_prepare_features.py and the approved methodology)
PERIODS = {
    "train": ("2018-01-01", "2024-12-31"),
    "validation": ("2025-01-01", "2025-12-31"),
    "test": ("2026-01-01", "2026-10-31"),
}
BASELINE_LAGS = ["lag_7", "lag_14", "lag_21", "lag_28"]  # fallback priority order
MIN_LAG = 5                  # youngest consumption lag allowed at the cutoff
MODEL_NAME = "baseline_lag7" # label used in exported files (agree with Person 4)

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("pandas:", pd.__version__, "| numpy:", np.__version__)
print("Repository root:", ROOT)
print("Features file:", FEATURES_PATH)

Matplotlib is building the font cache; this may take a moment.


Python: 3.11.9
Executable: /Users/mariegrandfils/Library/CloudStorage/OneDrive-Personal/HSG/Bachelor Economics/DSF/DSF_Group_Project/DSF_forecasting_electricity_consumption_Basel/.venv/bin/python
pandas: 3.1.1 | numpy: 2.4.6
Repository root: /Users/mariegrandfils/Library/CloudStorage/OneDrive-Personal/HSG/Bachelor Economics/DSF/DSF_Group_Project/DSF_forecasting_electricity_consumption_Basel
Features file: /Users/mariegrandfils/Library/CloudStorage/OneDrive-Personal/HSG/Bachelor Economics/DSF/DSF_Group_Project/DSF_forecasting_electricity_consumption_Basel/data/processed/features.csv


## 2. Load and check the data

We load the shared feature table `data/processed/features.csv`, built by Person 1 with `04_prepare_features.py` from the frozen data snapshots. No data are downloaded or rebuilt in this notebook.

Before computing anything, we check that the file matches the approved methodology. Each check stops the notebook with an error if it fails, so problems cannot go unnoticed:

1. All expected columns are present.
2. No consumption lag younger than `lag_5` is in the file (leakage guard).
3. Dates are unique and no calendar day is missing.
4. Period labels match the approved windows (train 2018–2024, validation 2025, test January–October 2026).
5. Target and baseline are numeric.
6. Missing values in the target and predictors are counted.
7. The scale of the target is consistent with daily totals in MWh/day.

The 2026 test rows remain in the DataFrame for structural checks only. We do not inspect their consumption values.

In [2]:
# Load the shared feature table (no downloading, no rebuilding)
df = pd.read_csv(FEATURES_PATH, parse_dates=["date"])

print("Shape (rows, columns):", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Already sorted by date in the file:", df["date"].is_monotonic_increasing)

# Sort chronologically so every later step can rely on the order
df = df.sort_values("date").reset_index(drop=True)
df.head()

Shape (rows, columns): (3226, 25)
Date range: 2018-01-01 to 2026-10-31
Already sorted by date in the file: True


,date,period,consumption_mwh,baseline,baseline_uses_fallback,weekday,is_weekend,month,is_holiday,is_christmas_period,...,forecast_precipitation,heating_degrees,cooling_degrees,lag_5,lag_6,lag_7,lag_14,lag_21,lag_28,lag_mean_5_11
0,2018-01-01,train,3062.233554,3013.999981,False,0,0,1,1,1,...,3.6,7.995833,0.0,3619.523783,3108.278054,3013.999981,4104.511028,4154.091096,4165.497217,3452.380986
1,2018-01-02,train,3673.523352,3108.278054,False,1,0,1,0,1,...,2.6,9.583333,0.0,3654.663325,3619.523783,3108.278054,4164.270389,4201.483851,4164.774973,3393.723082
2,2018-01-03,train,3823.853387,3619.523783,False,2,0,1,0,1,...,11.4,5.637500,0.0,3607.930176,3654.663325,3619.523783,4138.920236,4192.526272,4230.119719,3345.755918
3,2018-01-04,train,3865.945042,3654.663325,False,3,0,1,0,1,...,28.6,5.962500,0.0,3299.041167,3607.930176,3654.663325,4065.268655,4167.656222,4188.976652,3331.210362
4,2018-01-05,train,3835.695680,3607.930176,False,4,0,1,0,1,...,7.0,5.350000,0.0,2999.361438,3299.041167,3607.930176,3943.700325,4098.675157,4181.488001,3328.971132


In [4]:
# Structural checks: stop the notebook if the data do not match the methodology
EXPECTED_COLUMNS = [
    "date", "period", "consumption_mwh", "baseline", "baseline_uses_fallback",
    "weekday", "is_weekend", "month", "is_holiday", "is_christmas_period",
    "lag_7_is_holiday", "lag_14_is_holiday",
    "forecast_temp_mean", "forecast_temp_min", "forecast_temp_max",
    "forecast_precipitation", "heating_degrees", "cooling_degrees",
    "lag_5", "lag_6", "lag_7", "lag_14", "lag_21", "lag_28", "lag_mean_5_11",
]

# 1. Columns
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
extra_cols = [c for c in df.columns if c not in EXPECTED_COLUMNS]
assert not missing_cols, f"Missing columns: {missing_cols}"
print("1. All", len(EXPECTED_COLUMNS), "expected columns present | extra columns:", extra_cols or "none")

# 2. No consumption lag younger than lag_5 (leakage guard)
lag_numbers = [int(c[4:]) for c in df.columns if c.startswith("lag_") and c[4:].isdigit()]
assert min(lag_numbers) >= MIN_LAG, f"Lag younger than lag_{MIN_LAG} found: {sorted(lag_numbers)}"
print("2. Consumption lags in the file:", sorted(lag_numbers), "-> youngest is lag_" + str(min(lag_numbers)))

# 3. Dates: unique and no gaps in the calendar
assert df["date"].is_unique, "Duplicate target dates found"
full_calendar = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
missing_days = full_calendar.difference(df["date"])
print("3. Dates unique: True | missing calendar days:", len(missing_days))
if len(missing_days) > 0:
    print("   Missing days:", list(missing_days.date))

# 4. Period labels match the methodology
assert set(df["period"].unique()) == set(PERIODS), f"Unexpected period labels: {df['period'].unique()}"
for name, (start, end) in PERIODS.items():
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    in_window = df["date"].between(start, end)
    assert (df.loc[in_window, "period"] == name).all(), f"Dates in the {name} window carry another label"
    assert df.loc[df["period"] == name, "date"].between(start, end).all(), f"{name} rows outside their window"
print("4. Period labels match the methodology windows")
display(df.groupby("period", sort=False)["date"].agg(first="min", last="max", days="count"))

# 5. Data types of target and baseline
for col in ["consumption_mwh", "baseline"]:
    assert pd.api.types.is_numeric_dtype(df[col]), f"{col} is not numeric"
print("5. consumption_mwh and baseline are numeric:", df["consumption_mwh"].dtype, "/", df["baseline"].dtype)

# 6. Missing values (target and predictors)
missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0]
print("6. Missing values per column:", "none" if missing_counts.empty else "")
if not missing_counts.empty:
    display(missing_counts)

# 7. Target scale (units check), training and validation rows only
print("7. Daily consumption (MWh/day), train and validation only:")
display(
    df[df["period"] != "test"]
    .groupby("period", sort=False)["consumption_mwh"]
    .describe()
    .round(0)
)

1. All 25 expected columns present | extra columns: none
2. Consumption lags in the file: [5, 6, 7, 14, 21, 28] -> youngest is lag_5
3. Dates unique: True | missing calendar days: 0
4. Period labels match the methodology windows


,first,last,days
period,,,
train,2018-01-01,2024-12-31,2557
validation,2025-01-01,2025-12-31,365
test,2026-01-01,2026-10-31,304


5. consumption_mwh and baseline are numeric: float64 / float64
6. Missing values per column: none
7. Daily consumption (MWh/day), train and validation only:


,count,mean,std,min,25%,50%,75%,max
period,,,,,,,,
train,2557.0,3463.0,351.0,2456.0,3213.0,3505.0,3725.0,4455.0
validation,365.0,3323.0,298.0,2619.0,3109.0,3350.0,3548.0,3968.0


## 3. Baseline formula and fallback

For target day $d$, the baseline uses the most recent same-weekday observation permitted by the five-day rule, in this priority order:

1. `lag_7` ($y_{d-7}$)
2. if missing: `lag_14` ($y_{d-14}$)
3. if missing: `lag_21` ($y_{d-21}$)
4. if missing: `lag_28` ($y_{d-28}$)

All four candidates lie at least five days before the target date, so each one respects the availability assumption. A fallback only ever goes further back in time and never uses information from after the forecast cutoff.

**If all four lags are missing**, no same-weekday observation from the past four weeks exists. Instead of inventing a value, the notebook stops and lists the affected dates, so that the group can agree on a treatment within the common validation framework. Any such treatment must use only observations available at each date's forecast cutoff, never full-sample averages.

**What we verify below:**
- We recompute the baseline from the lag columns and compare it with the `baseline` column saved in `features.csv`. If they differed, we would investigate rather than overwrite the file.
- We create `baseline_lag_used`, recording which lag produced each prediction, and check it against the saved indicator `baseline_uses_fallback`.
- We report how often each lag was used, per period.

**Note on the fallback in practice:** `04_prepare_features.py` stops without saving if any feature column contains a missing value, including `lag_7`. A saved `features.csv` therefore cannot contain rows that need a fallback. The recomputation below verifies the baseline and acts as a safeguard should the data ever change.

In [5]:
# Recompute the baseline from the lag columns, in priority order
assert all(int(lag[4:]) >= MIN_LAG for lag in BASELINE_LAGS), "Baseline uses a forbidden lag"

recomputed = pd.Series(np.nan, index=df.index)
df["baseline_lag_used"] = "none"

for lag in BASELINE_LAGS:
    use_this_lag = recomputed.isna() & df[lag].notna()
    recomputed[use_this_lag] = df.loc[use_this_lag, lag]
    df.loc[use_this_lag, "baseline_lag_used"] = lag

# Final rule: if all four lags are missing, stop and list the dates (no value is invented)
no_prediction = df["baseline_lag_used"] == "none"
if no_prediction.any():
    print("Dates without any same-weekday lag:", list(df.loc[no_prediction, "date"].dt.date))
    raise ValueError("Baseline cannot be computed for some dates. Agree on a treatment with the group first.")
print("Dates where all four lags are missing:", no_prediction.sum())

# Compare with the baseline column saved by Person 1
difference = (recomputed - df["baseline"]).abs()
n_different = (difference > 1e-9).sum()
print("Rows where recomputed and saved baseline differ:", n_different)
if n_different > 0:
    display(df.loc[difference > 1e-9, ["date", "period", "baseline"] + BASELINE_LAGS])
assert n_different == 0, "Recomputed baseline differs from features.csv: investigate before continuing"

# Compare with the saved fallback indicator
fallback_recomputed = df["baseline_lag_used"] != "lag_7"
assert (fallback_recomputed == df["baseline_uses_fallback"]).all(), "Fallback indicator differs from features.csv"
print("Saved fallback indicator matches the recomputed one: True")

# How often each lag was used, per period (zeros shown explicitly)
lag_usage = (
    pd.crosstab(df["period"], df["baseline_lag_used"])
    .reindex(index=list(PERIODS), columns=BASELINE_LAGS + ["none"], fill_value=0)
)
display(lag_usage)

n_val = (df["period"] == "validation").sum()
n_val_fallback = (fallback_recomputed & (df["period"] == "validation")).sum()
print(f"Validation 2025: {n_val - n_val_fallback} of {n_val} predictions use lag_7; "
      f"fallback used {n_val_fallback} times.")

Dates where all four lags are missing: 0
Rows where recomputed and saved baseline differ: 0
Saved fallback indicator matches the recomputed one: True


baseline_lag_used,lag_7,lag_14,lag_21,lag_28,none
period,,,,,
train,2557,0,0,0,0
validation,365,0,0,0,0
test,304,0,0,0,0


Validation 2025: 365 of 365 predictions use lag_7; fallback used 0 times.


## 4. Evaluation periods

| Period | Dates | Role in this notebook |
|---|---|---|
| Training | 2018-01-01 – 2024-12-31 | Not needed: the baseline has no parameters to estimate |
| Validation | 2025-01-01 – 2025-12-31 | **Evaluated here** (365 days, 12 monthly folds) |
| Test (holdout) | 2026-01-01 – 2026-10-31 | Not evaluated; reserved for the final comparison coordinated by Person 4 |

The baseline is evaluated on every day of 2025, using the same target dates and units (MWh/day) as the linear models in Notebook 06. Each prediction is labelled with its validation month (`2025-01` … `2025-12`), so the results line up with the 12 monthly validation folds. Because the baseline does not estimate anything, monthly refitting does not change its predictions.

In [6]:
# Select the 2025 validation rows: the only rows evaluated in this notebook
val = df[df["period"] == "validation"].copy()

# Every day of 2025 must appear exactly once, with an observed target and a valid prediction
start, end = PERIODS["validation"]
expected_dates = pd.date_range(start, end, freq="D")
assert len(val) == len(expected_dates) and val["date"].isin(expected_dates).all(), \
    "Validation rows do not cover every day of 2025 exactly once"
assert val["consumption_mwh"].notna().all(), "Missing actual consumption in 2025"
assert np.isfinite(val["baseline"]).all(), "Missing or non-finite baseline predictions in 2025"

# Label each prediction with its validation month (matches the 12 monthly folds)
val["validation_month"] = val["date"].dt.strftime("%Y-%m")

print("Evaluated period:", val["date"].min().date(), "to", val["date"].max().date())
print("Evaluated dates:", len(val), "| validation months:", val["validation_month"].nunique())
print("Test rows kept aside (not evaluated here):", (df["period"] == "test").sum())

Evaluated period: 2025-01-01 to 2025-12-31
Evaluated dates: 365 | validation months: 12
Test rows kept aside (not evaluated here): 304


## 5. Baseline performance on the 2025 validation period

**Error convention.** For each target day, the prediction error is

$$e_d = y_d - \hat y_d$$

A positive error means the baseline predicted too little consumption; a negative error means it predicted too much.

**Mean Absolute Error (primary metric):**

$$MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat y_i|$$

the average size of a forecast error in MWh/day. It is directly interpretable for a procurement team.

**Root Mean Squared Error (secondary metric):**

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat y_i)^2}$$

Squaring gives large errors more weight, so RMSE is always at least as large as MAE. A large gap between the two indicates that a few days have very large errors.

**Mean error** (the average of $e_d$) is reported as a diagnostic only: it shows whether the baseline systematically over- or underpredicts. It is not used to select models.

In [7]:
# Metrics function, reused in Notebook 06 so that all models are scored identically
from sklearn.metrics import mean_absolute_error, root_mean_squared_error


def compute_metrics(actual, predicted):
    """Return the number of days, MAE, RMSE and mean error (all in MWh/day)."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    assert len(actual) == len(predicted) > 0, "Empty or misaligned inputs"
    assert np.isfinite(actual).all() and np.isfinite(predicted).all(), "Missing or non-finite values"
    return {
        "n_days": len(actual),
        "mae_mwh": mean_absolute_error(actual, predicted),
        "rmse_mwh": root_mean_squared_error(actual, predicted),
        "mean_error_mwh": np.mean(actual - predicted),
    }


# Prediction columns, named as they will appear in the exported table
val["actual_mwh"] = val["consumption_mwh"]
val["predicted_mwh"] = val["baseline"]
val["prediction_error_mwh"] = val["actual_mwh"] - val["predicted_mwh"]
val["absolute_error_mwh"] = val["prediction_error_mwh"].abs()

baseline_overall = pd.DataFrame(
    [{"model": MODEL_NAME, **compute_metrics(val["actual_mwh"], val["predicted_mwh"])}]
)
display(baseline_overall.round(1))

# Cross-check: the formulas from the Markdown cell, computed by hand, give the same values
manual_mae = val["absolute_error_mwh"].mean()
manual_rmse = np.sqrt((val["prediction_error_mwh"] ** 2).mean())
assert np.isclose(manual_mae, baseline_overall.loc[0, "mae_mwh"])
assert np.isclose(manual_rmse, baseline_overall.loc[0, "rmse_mwh"])
print("Check: formulas computed by hand give the same MAE and RMSE: True")

,model,n_days,mae_mwh,rmse_mwh,mean_error_mwh
0,baseline_lag7,365,137.3,201.5,0.4


Check: formulas computed by hand give the same MAE and RMSE: True
